In [1]:
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
from torch.optim import SGD
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import networkx as nx
import time
import math
import copy
import os

torch.manual_seed(0)
np.random.seed(0)

In [2]:
from os import chdir
from pathlib import Path

new_path = Path("/Users/haowen/Documents/Decentral RS/fed-learning-main")

if new_path.exists():
    os.chdir(new_path)
    print(f"Working directory changed to: {Path.cwd()}")
else:
    print("Path does not exist.")


Working directory changed to: /Users/haowen/Documents/Decentral RS/fed-learning-main


In [3]:
from src.models.MatrixFactorization import MF, UMF
from src.graphs import random_k_out_graph, create_graph
from src.users import User
from src.training.decentralized import (decentralized_train_n_epochs,
                                        decentralized_validate_loop)
from src.data_utils import create_batched_dataloaders, create_dataloader

In [4]:
#Make data sample iterable
from torch.utils.data import Dataset, DataLoader, TensorDataset, Sampler
import torch
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.optim import SGD

from collections import Counter
import networkx as nx
from networkx.generators.classic import empty_graph
from networkx.utils import discrete_sequence, py_random_state, weighted_choice

import seaborn as sns
from sklearn.utils import shuffle

## Parameter Setting

In [6]:
model = "umf"
val_loader_type = "rs"
train_loader_type = "rs"
userprop = None
n_factors = 30
sparse = False
batch_size = 10
lr = 0.03871364416669273
weight_decay = 0.14214480688557163
mom = 0.001
graph_seed = 1
n_epochs = 50

## Main

In [43]:
SAMPLED_DATA_DIR       = r"/Users/haowen/Documents/Decentral RS/rebuttal/code/8"


train_path = os.path.join(SAMPLED_DATA_DIR, f"hm_subset_train.csv")
test_path   = os.path.join(SAMPLED_DATA_DIR, f"hm_subset_test.csv")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
train_df.head()

,user_id,item_id,bought
0,850,206,1
1,520,7,1
2,630,2196,1
3,46,1145,1
4,1078,680,1


In [45]:
n_users = len(user_enc)
n_items = len(item_enc)
print(f"Total Users: {n_users}")
print(f"Total Items: {n_items}")

Total User: 1760
Total Item: 2881


In [47]:
train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=0)
train_data_loader = create_dataloader(
        df=train_df, dl_type=train_loader_type, batch_size=batch_size, p=userprop
    )
val_data_loader = create_dataloader(df=val_df, dl_type=val_loader_type)
test_data_loader = create_dataloader(df=test_df, dl_type=val_loader_type)


In [49]:
users = {}
for i in tqdm(range(n_users)):
    # model = MF(n_users=n_users, n_items=n_items)
    user_model = UMF(n_items, n_factors = n_factors, sparse = sparse)
    # model = GeneralizedMFOneLayer(n_users=n_users, n_items=n_items)
    #users[i] = User(id=i, model=user_model, optimizer=SGD(user_model.parameters(), lr=lr, weight_decay=weight_decay), model_name = model)
    users[i] = User(
            id=i,
            model=user_model,
            optimizer=SGD(user_model.parameters(), lr=lr, weight_decay=weight_decay, momentum=mom),
            model_name=model,
        )

  0%|          | 0/1760 [00:00<?, ?it/s]

In [50]:
graph = random_k_out_graph(n=n_users, k=2, seed=graph_seed)

In [51]:
import torch
torch.manual_seed(0)
train_losses, val_losses, time_per_epoch, commutes = decentralized_train_n_epochs(
        user_models=users,
        train_loader=train_data_loader,
        val_loader=val_data_loader,
        epochs=n_epochs,
        graph=graph,
    )

  0%|          | 0/62568 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 2.0379 | Validation Loss: 1.7139 | Time Elapsed: 40.506892 sec |Commute: 125031 | Commute 
Cost: 5588010648

  0%|          | 0/62568 [00:00<?, ?it/s]

Epoch 2 | Train Loss: 0.5159 | Validation Loss: 1.1098 | Time Elapsed: 44.599669 sec |Commute: 125031 | Commute 
Cost: 5588010648

  0%|          | 0/62568 [00:00<?, ?it/s]

...

Epoch 16 | Train Loss: 0.6085 | Validation Loss: 0.5560 | Time Elapsed: 30.097528 sec |Commute: 125031 | Commute 
Cost: 5588010648

  0%|          | 0/62568 [00:00<?, ?it/s]

Epoch 17 | Train Loss: 0.6112 | Validation Loss: 0.5471 | Time Elapsed: 40.105665 sec |Commute: 125031 | Commute 
Cost: 5588010648

Early stopping.

Total time elapsed: 684.2965109581128

In [53]:
test_loss = decentralized_validate_loop(users, test_data_loader)

In [54]:
test_loss

0.53085965